In [11]:
import os
from dotenv import load_dotenv

# disbling tracing before importing agents
os.environ["OPENAI_AGENTS_DISABLE_TRACING"] = "1"

from agents import Agent, Runner, function_tool, OpenAIChatCompletionsModel
from openai import  AsyncOpenAI

sequential Agents

In [26]:
@function_tool
def get_user_id(email: str) -> str:
    """Get user ID from email"""
    return "user_123"

@function_tool
def get_user_orders(user_id: str) -> str:
    """Get the user's order based on the user ID"""
    return "Order #1: Laptop Order #2: Mouse"

@function_tool
def get_order_status(order_id: str) -> str:
#def get_order_status(userid: str) -> str:
    """Get the status of an order based on the user ID"""
    return "Shipped - Arriving Tomorrow"

In [ ]:
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
if openai_api_key:
    print("API KEY FOUND")
else:
    print("API KEY NOT FOUND")

API KEY FOUND


In [19]:
client = AsyncOpenAI(api_key=openai_api_key,base_url="https://openrouter.ai/api/v1")
Model = OpenAIChatCompletionsModel(openai_client=client,model="gpt-4o-mini")

In [20]:
system_message = (
    """
    You are a sales aisstant,you help users check the details of their orders,use the get_user_id,get_user_orders,get_order_status tools whenever a customer ask for assistance
    """
)

sales_agent = Agent(
    name="Sales Agent",
    instructions = system_message,
    model=Model,
    tools=[get_user_id,get_user_orders,get_order_status]
)

In [28]:
user_prompt = input("Enter Your Prompt")
runner = await Runner.run(sales_agent,user_prompt)
print(runner.final_output)

The status of your order is "Shipped" and it is arriving tomorrow.


A Parrallel and Sequential Agent

In [38]:
@function_tool
def get_weather(city: str) -> str:
    """Get weather for a city"""
    return f"{city}: Sunny, 25C"

@function_tool
def get_news(topic: str) -> str:
    """Get latest news on a topic"""
    return f"The latest on {topic} new"

@function_tool
def get_stock_price(symbol: str) -> str:
    """Get the stock prices of stock"""
    return f"{symbol}: $150.00"

In [ ]:
system_instruct = ("You are a personal asssistant to the users,you help him get things done base online,use the tools the tools available based on the user request especially concerning their morning briefing")
personal_agent = Agent(
    name="Personal Agent",
    instructions=system_instruct,
    model=Model,
    tools=[get_weather,get_news,get_stock_price]
)

In [40]:
prompt = input("How can help you")
agent_runner = await Runner.run(personal_agent,prompt)
print(agent_runner.final_output)

Here’s your morning briefing:

1. **Elon Musk Scandal**: The latest news on the Elon Musk scandal is ongoing and developing. (For detailed updates, please specify if you want more specific information on a particular aspect of the scandal.)

2. **Apple Stock Price**: The current stock price for Apple Inc. (AAPL) is $150.00.

3. **Weather in Abuja**: Today's weather in Abuja is sunny with a temperature of 25°C. 

If you need further details on any of these topics, feel free to ask!


conditional Agent

In [41]:
@function_tool
def search_documents(query: str) -> str:
    """Search uploaded documents"""
    return f"Document result for '{query}':...."

@function_tool
def search_web(query: str) -> str:
    """Search the web for relevant information"""
    return f"online result for the '{query}':...."

@function_tool
def search_database(query: str) -> str:
    """Search database """
    return f"database result for '{query}':...."



In [42]:
system_prompt = ("""
You are a search assistant.
    
    Choose the right search based on the query:
    - For current events/general info → use search_web
    - For company/employee info → use search_database
    - For policy/procedure questions → use search_documents
""")

smarta = Agent(
    name="Smarta",
    instructions=system_prompt,
    model=Model,
    tools=[search_documents,search_web, search_database]
)

In [43]:
user_input = input("Enter Prompt")
runner = await Runner.run(smarta,user_input)
print(runner.final_output)

I've found the document regarding the company's policy on taking holidays. Please refer to the company policy documents for detailed information about holiday entitlements, procedures for requesting time off, and any other relevant guidelines. If you need more specific details or have further inquiries, please let me know!
